In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as sk

In [2]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import randint, uniform

In [3]:
df = pd.read_csv("insurance.csv")
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


### Кодирование категориальных признаков

In [4]:
df =  pd.get_dummies(df, drop_first=True)

In [5]:
df

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,False,True,False,False,True
1,18,33.770,1,1725.55230,True,False,False,True,False
2,28,33.000,3,4449.46200,True,False,False,True,False
3,33,22.705,0,21984.47061,True,False,True,False,False
4,32,28.880,0,3866.85520,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,True,False,True,False,False
1334,18,31.920,0,2205.98080,False,False,False,False,False
1335,18,36.850,0,1629.83350,False,False,False,True,False
1336,21,25.800,0,2007.94500,False,False,False,False,True


### Разделение на признаков от цели и Тренировочную на тестовую

In [6]:
X = df.drop('charges', axis=1)
y = df['charges']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y ,test_size=0.2, random_state=42
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (1070, 8)
Размер тестовой выборки: (268, 8)


### Стандартизация

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучение на моделях машинного обучения

## Линейная регрессия

In [9]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Среднеквадратичная ошибка (MSE): {mse_lr:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_lr}")
print(f"Коэффициент детерминации (R²): {r2_lr:.4f}")

Среднеквадратичная ошибка (MSE): 33596915.8514
Среднеабсолютная ошибка (MAE): 4181.194473753654
Коэффициент детерминации (R²): 0.7836


## Полиномиальная регрессия

In [10]:
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

y_pred_poly = poly_model.predict(X_test_poly)

mse_poly = mean_squared_error(y_test, y_pred_poly)
mae_poly = mean_absolute_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

print(f"Среднеквадратичная ошибка (MSE): {mse_poly:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_poly}")
print(f"Коэффициент детерминации (R²): {r2_poly:.4f}")

Среднеквадратичная ошибка (MSE): 20712805.9879
Среднеабсолютная ошибка (MAE): 2729.5001336394525
Коэффициент детерминации (R²): 0.8666


## Деревья решений

In [11]:
dt_model = DecisionTreeRegressor(
    max_depth=10,           # Максимальная глубина дерева
    min_samples_split=20,   # Минимальное количество образцов для разделения узла
    min_samples_leaf=10,    # Минимальное количество образцов в листе
    random_state=42
)

dt_model.fit(X_train_scaled, y_train)

y_pred_dt = dt_model.predict(X_test_scaled)

mse_dt = mean_squared_error(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print(f"Среднеквадратичная ошибка (MSE): {mse_dt:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_dt:.4f}")
print(f"Коэффициент детерминации (R²): {r2_dt:.4f}")

Среднеквадратичная ошибка (MSE): 22637166.0979
Среднеабсолютная ошибка (MAE): 2751.7451
Коэффициент детерминации (R²): 0.8542


## Случайный лес

In [12]:
rf_model = RandomForestRegressor(
    n_estimators=100,       # Количество деревьев
    max_depth=None,         # Максимальная глубина (None - без ограничения)
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)

mse_rf = mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Среднеквадратичная ошибка (MSE): {mse_rf:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_rf:.4f}")
print(f"Коэффициент детерминации (R²): {r2_rf:.4f}")

Среднеквадратичная ошибка (MSE): 20864569.5134
Среднеабсолютная ошибка (MAE): 2543.9758
Коэффициент детерминации (R²): 0.8656


## Метод опорных векторов

In [13]:
svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr_model.fit(X_train_scaled, y_train)

y_pred_svr = svr_model.predict(X_test_scaled)

mse_svr = mean_squared_error(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print(f"Среднеквадратичная ошибка (MSE): {mse_svr:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_svr:.4f}")
print(f"Коэффициент детерминации (R²): {r2_svr:.4f}")

Среднеквадратичная ошибка (MSE): 166128803.8085
Среднеабсолютная ошибка (MAE): 8612.4084
Коэффициент детерминации (R²): -0.0701


## K - ближайших соседей

In [14]:
knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred_knn = knn_model.predict(X_test_scaled)

mse_knn = mean_squared_error(y_test, y_pred_knn)
mae_knn = mean_absolute_error(y_test, y_pred_knn)
r2_knn = r2_score(y_test, y_pred_knn)

print(f"Среднеквадратичная ошибка (MSE): {mse_knn:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_knn:.4f}")
print(f"Коэффициент детерминации (R²): {r2_knn:.4f}")

Среднеквадратичная ошибка (MSE): 30459865.8232
Среднеабсолютная ошибка (MAE): 3494.7461
Коэффициент детерминации (R²): 0.8038


## Градиентный бустинг

In [15]:
from catboost import CatBoostRegressor
import xgboost as xgb
import lightgbm as lgb

### CatBoost

In [16]:
model_cat = CatBoostRegressor(
    iterations=500, 
    learning_rate=0.5, 
    depth=6
)

model_cat.fit(X_train_scaled, y_train, eval_set=(X_test_scaled, y_test))

predications_cat = model_cat.predict(X_test_scaled)

mse_cat = mean_squared_error(y_test, predications_cat)
mae_cat = mean_absolute_error(y_test, predications_cat)
r2_cat = r2_score(y_test, predications_cat)

print(f"Среднеквадратичная ошибка (MSE): {mse_cat:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_cat:.4f}")
print(f"Коэффициент детерминации (R²): {r2_cat:.4f}")

0:	learn: 7571.2270970	test: 7719.2213348	best: 7719.2213348 (0)	total: 137ms	remaining: 1m 8s
1:	learn: 5709.3050160	test: 5696.0780986	best: 5696.0780986 (1)	total: 138ms	remaining: 34.3s
2:	learn: 5055.2533878	test: 4955.1178825	best: 4955.1178825 (2)	total: 139ms	remaining: 23s
3:	learn: 4711.6873069	test: 4589.8479339	best: 4589.8479339 (3)	total: 140ms	remaining: 17.4s
4:	learn: 4535.4819623	test: 4400.7946600	best: 4400.7946600 (4)	total: 142ms	remaining: 14s
5:	learn: 4432.5048360	test: 4359.2620139	best: 4359.2620139 (5)	total: 143ms	remaining: 11.8s
6:	learn: 4391.9380656	test: 4347.2675501	best: 4347.2675501 (6)	total: 145ms	remaining: 10.2s
7:	learn: 4353.7614025	test: 4354.1050058	best: 4347.2675501 (6)	total: 146ms	remaining: 8.98s
8:	learn: 4324.4234456	test: 4352.2639390	best: 4347.2675501 (6)	total: 147ms	remaining: 8.03s
9:	learn: 4278.6752658	test: 4313.3192570	best: 4313.3192570 (9)	total: 148ms	remaining: 7.28s
10:	learn: 4248.7429464	test: 4309.1571437	best: 4309.

### XGBoost

In [17]:
model_xg = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate = 0.5,
    max_depth = 6
)

model_xg.fit(X_train_scaled, y_train)

predications_xg = model_xg.predict(X_test_scaled)

mse_xg = mean_squared_error(y_test, predications_xg)
mae_xg = mean_absolute_error(y_test, predications_xg)
r2_xg = r2_score(y_test, predications_xg)

print(f"Среднеквадратичная ошибка (MSE): {mse_xg:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_xg:.4f}")
print(f"Коэффициент детерминации (R²): {r2_xg:.4f}")

Среднеквадратичная ошибка (MSE): 25210144.7632
Среднеабсолютная ошибка (MAE): 2839.4197
Коэффициент детерминации (R²): 0.8376


## Random Search для XGBoost

In [18]:
# Определяем распределения параметров
param_distributions = {
    'n_estimators': randint(50, 500),        # Случайное целое от 50 до 500
    'max_depth': randint(3, 12),             # Случайное целое от 3 до 12
    'learning_rate': uniform(0.01, 0.3),     # Случайное от 0.01 до 0.31
    'subsample': uniform(0.6, 0.4),          # Случайное от 0.6 до 1.0
    'colsample_bytree': uniform(0.6, 0.4)    # Случайное от 0.6 до 1.0
}

# Создаём Random Search
random_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=42),
    param_distributions=param_distributions,
)

# Запускаем поиск
random_search.fit(X_train, y_train)

# Лучшие параметры
print(f"\nЛучшие параметры: {random_search.best_params_}")

# Проверка на тесте
best_xgb = random_search.best_estimator_
y_pred = best_xgb.predict(X_test)

mse_xg_rd = mean_squared_error(y_test, y_pred)
mae_xg_rd = mean_absolute_error(y_test, y_pred)
r2_xg_rd = r2_score(y_test, y_pred)

print(f"Среднеквадратичная ошибка (MSE): {mse_xg_rd:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_xg_rd:.4f}")
print(f"Коэффициент детерминации (R²): {r2_xg_rd:.4f}")


Лучшие параметры: {'colsample_bytree': np.float64(0.882894201362418), 'learning_rate': np.float64(0.06770248591533715), 'max_depth': 3, 'n_estimators': 95, 'subsample': np.float64(0.787558247884794)}
Среднеквадратичная ошибка (MSE): 18369894.2292
Среднеабсолютная ошибка (MAE): 2444.4438
Коэффициент детерминации (R²): 0.8817


### LightGBM

In [19]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=500, 
    learning_rate = 0.5,
    max_depth = 6
)

lgb_model.fit(X_train, y_train)

predications_lgb = lgb_model.predict(X_test)

mse_lgb = mean_squared_error(y_test, predications_lgb)
mae_lgb = mean_absolute_error(y_test, predications_lgb)
r2_lgb = r2_score(y_test, predications_lgb)

print(f"Среднеквадратичная ошибка (MSE): {mse_lgb:.4f}")
print(f"Среднеабсолютная ошибка (MAE): {mae_lgb:.4f}")
print(f"Коэффициент детерминации (R²): {r2_lgb:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000275 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 1070, number of used features: 8
[LightGBM] [Info] Start training from score 13346.089733
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

# Обучение на моделях нейронной сети

In [20]:
from fastai.tabular.all import *
import torch.nn.functional as F

c:\Users\dimam\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## MLP

In [ ]:
# Разделение признаков
dep_var = 'charges'
cont_names = ['age', 'bmi']
cat_names = ['sex', 'children', 'smoker', 'region']

# Логарифмирование целевой переменной (для стабильности обучения)
df[dep_var] = np.log1p(df[dep_var])

# Разделение на train/valid
splits = RandomSplitter(valid_pct=0.2, seed=42)(range_of(df))

# TabularPandas
to = TabularPandas(
    df,
    procs=[FillMissing, Normalize],
    cat_names=cat_names,
    cont_names=cont_names,
    y_names=dep_var,
    splits=splits
)

dls = to.dataloaders(bs=64)
print(f"Количество непрерывных признаков: {len(dls.cont_names)}")

Could not do one pass in your dataloader, there is something wrong in it. Please see the stack trace below:


KeyError: "None of [Index(['sex', 'smoker', 'region'], dtype='object')] are in the [columns]"

In [ ]:
learn_mlp = tabular_learner(
    dls,
    layers=[200, 100],       # два скрытых слоя
    metrics=[rmse, mae],
    y_range=None
)

learn_mlp.lr_find()
learn_mlp.fit_one_cycle(20, lr_max=1e-2)

## RNN